# Feature Ablation Training

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 73.


In [ ]:
# ============================================================
# CONTROLLED FEATURE ABLATION — CONCAT-ONLY GCN
# TRAIN-ONLY FEATURE PREPROCESSING
# CORRECTED GRAPH RECIPE
# ============================================================
#
# CHANGE ONLY:
#
#   FEATURE_MODE
#   SEED
#
# FEATURE_MODE:
#   "morgan_only"
#   "rdkit_only"
#   "morgan_rdkit"
#
# SEED:
#   42
#   43
#   44
#
# ------------------------------------------------------------
# Fixed across ALL runs:
#
# - Exact original Interaction train/cal/test splits
# - Exact 84-class label space
# - Train-only prepared features from Cell A
# - GCN encoder: input -> 384 -> 384 -> 192
# - BN after each GCN layer
# - ReLU after first two GCN layers
# - No final encoder activation
# - Ordered concat: z(Drug1) || z(Drug2)
# - Classifier: 384 -> 512 -> 256 -> 84
# - Dropout = 0.35
# - Weighted CrossEntropyLoss
# - AdamW
# - LR = 7e-4
# - Weight decay = 1e-4
# - Batch size = 512
# - Max epochs = 120
# - Early stopping patience = 18
# - Scheduler factor = 0.5
# - Scheduler patience = 5
# - Checkpoint selection by Calibration Macro-F1 (84)
# - Test used ONLY after model selection
#
# ------------------------------------------------------------
# CORRECTED GRAPH RECIPE:
#
# 1. Start from Train ordered interaction rows.
# 2. Deduplicate ORIGINAL ordered Train pairs:
#       134,249 -> 134,122
# 3. Add reverse edge for every unique ordered edge.
# 4. DO NOT globally deduplicate after reversal.
#
# Final graph:
#       134,122 * 2 = 268,244 directed edges
#
# Reciprocal overlap is retained deliberately.
#
# ============================================================


# ============================================================
# 0) ONLY CHANGE THESE TWO VALUES
# ============================================================

FEATURE_MODE = "morgan_only"
SEED = 42


# ============================================================
# 1) IMPORTS
# ============================================================

import os

os.environ.setdefault(
    "CUBLAS_WORKSPACE_CONFIG",
    ":4096:8",
)

import sys
import json
import random
import hashlib
import zipfile
import subprocess
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch import nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    f1_score,
)


# ============================================================
# 2) TORCH GEOMETRIC
# ============================================================

if importlib.util.find_spec(
    "torch_geometric"
) is None:

    print(
        "Installing torch-geometric...",
        flush=True,
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "torch-geometric==2.8.0.post1",
        ],
        check=True,
        timeout=300,
    )


from torch_geometric.nn import GCNConv
import torch_geometric


print(
    "torch:",
    torch.__version__,
    flush=True,
)

print(
    "torch_geometric:",
    torch_geometric.__version__,
    flush=True,
)


# ============================================================
# 3) BASIC CONFIG QA
# ============================================================

VALID_FEATURE_MODES = {
    "morgan_only",
    "rdkit_only",
    "morgan_rdkit",
}

VALID_SEEDS = {
    42,
    43,
    44,
}


assert FEATURE_MODE in VALID_FEATURE_MODES, (
    f"Invalid FEATURE_MODE: {FEATURE_MODE}"
)

assert SEED in VALID_SEEDS, (
    f"Invalid SEED: {SEED}"
)


# ============================================================
# 4) PATHS
# ============================================================

ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/ddi-v2-menna-projectt"
)


FEATURE_ROOT = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING"
)


RUN_ROOT = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_CONTROLLED_V1"
)


FEATURE_FILES = {

    "morgan_only":
        FEATURE_ROOT
        /
        "molecular_features_morgan_only.npz",

    "rdkit_only":
        FEATURE_ROOT
        /
        "molecular_features_rdkit_only.npz",

    "morgan_rdkit":
        FEATURE_ROOT
        /
        "molecular_features_morgan_rdkit.npz",
}


FEATURE_PATH = (
    FEATURE_FILES[
        FEATURE_MODE
    ]
)


TRAIN_PATH = (
    ROOT
    /
    "clean_train_split.csv"
)


CAL_PATH = (
    ROOT
    /
    "clean_calibration_split.csv"
)


TEST_PATH = (
    ROOT
    /
    "clean_test_split.csv"
)


FEATURE_META_PATH = (
    FEATURE_ROOT
    /
    "feature_ablation_metadata.json"
)


OUT = (
    RUN_ROOT
    /
    FEATURE_MODE
    /
    f"seed_{SEED}"
)


OUT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 5) FIXED EXPERIMENT CONFIG
# ============================================================

CONFIG = {

    "experiment":
        "controlled_feature_ablation_train_only",

    "feature_mode":
        FEATURE_MODE,

    "seed":
        SEED,

    "model":
        "concat_only_gcn",

    "graph":
        "TRAIN_ORDERED_UNIQUE_THEN_REVERSE",

    "graph_forward_unique_edges":
        134122,

    "graph_edge_count":
        268244,

    "reciprocal_duplicates_after_reverse":
        "retained",

    "graph_edge_weight":
        False,

    "encoder":
        [
            "input",
            384,
            384,
            192,
        ],

    "encoder_batch_norm":
        True,

    "encoder_relu":
        "after GCN1 and GCN2 only",

    "encoder_final_activation":
        False,

    "pair_operator":
        "ordered_concat_z1_z2",

    "classifier":
        [
            384,
            512,
            256,
            84,
        ],

    "dropout":
        0.35,

    "batch_size":
        512,

    "max_epochs":
        120,

    "early_stopping_patience":
        18,

    "optimizer":
        "AdamW",

    "learning_rate":
        0.0007,

    "weight_decay":
        0.0001,

    "scheduler":
        "ReduceLROnPlateau",

    "scheduler_mode":
        "max",

    "scheduler_factor":
        0.5,

    "scheduler_patience":
        5,

    "loss":
        "weighted CrossEntropyLoss",

    "class_weights":
        (
            "sqrt(max_count/(count+1)); "
            "clip[0.5,3]; divide_by_mean"
        ),

    "model_selection":
        "Calibration Macro-F1 over 84 classes",

    "test_use":
        "final evaluation only",

    "training_encoder_scope":
        "Train drugs only",

    "evaluation_encoder_scope":
        (
            "all molecular feature rows, "
            "with Train interaction graph only"
        ),

    "heldout_graph_access":
        "self-loop only if absent from Train interaction graph",
}


# ============================================================
# 6) HELPERS
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def save_json(
    obj,
    path,
):

    temporary = path.with_name(
        path.name
        +
        ".tmp"
    )

    temporary.write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    os.replace(
        temporary,
        path,
    )


def save_checkpoint(
    obj,
    path,
):

    temporary = path.with_name(
        path.name
        +
        ".tmp"
    )

    torch.save(
        obj,
        temporary,
    )

    os.replace(
        temporary,
        path,
    )


def cpu_state(
    model,
):

    return {

        name:
            tensor
            .detach()
            .cpu()
            .clone()

        for name, tensor
        in model.state_dict().items()
    }


# ============================================================
# 7) INPUT FILE QA
# ============================================================

required_files = [

    FEATURE_PATH,

    TRAIN_PATH,

    CAL_PATH,

    TEST_PATH,

    FEATURE_META_PATH,
]


for path in required_files:

    assert path.is_file(), (
        f"Missing required file:\n{path}"
    )


assert torch.cuda.is_available(), (
    "Enable a Kaggle GPU before training."
)


device = torch.device(
    "cuda"
)


print("\n" + "=" * 90)
print("FEATURE ABLATION RUN")
print("=" * 90)

print(
    "Feature mode:",
    FEATURE_MODE,
)

print(
    "Seed:",
    SEED,
)

print(
    "GPU:",
    torch.cuda.get_device_name(0),
)


# ============================================================
# 8) LOAD FEATURE PREPROCESSING METADATA
# ============================================================

feature_meta = json.loads(
    FEATURE_META_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    feature_meta[
        "total_drugs"
    ]
    ==
    1705
)


assert (
    feature_meta[
        "train_fit_drugs"
    ]
    ==
    1675
)


assert (
    feature_meta[
        "calibration_unseen_drugs"
    ]
    ==
    14
)


assert (
    feature_meta[
        "test_unseen_drugs"
    ]
    ==
    20
)


assert (
    feature_meta[
        "rdkit_retained_train_only"
    ]
    ==
    201
)


print(
    " Train-only feature metadata verified.",
    flush=True,
)


# ============================================================
# 9) LOAD SELECTED FEATURE CACHE
# ============================================================

with np.load(
    FEATURE_PATH,
    allow_pickle=True,
) as saved:

    x_np = (
        saved[
            "x"
        ]
        .astype(
            np.float32
        )
    )

    all_drugs = (
        saved[
            "all_drugs"
        ]
        .astype(str)
    )

    fitted_train_indices = (
        saved[
            "train_drug_indices"
        ]
        .astype(
            np.int64
        )
    )


assert x_np.shape[0] == 1705

assert len(
    all_drugs
) == 1705

assert len(
    set(
        all_drugs.tolist()
    )
) == 1705

assert np.isfinite(
    x_np
).all()


INPUT_DIM = int(
    x_np.shape[1]
)


expected_dims = {

    "morgan_only":
        2048,

    "rdkit_only":
        201,

    "morgan_rdkit":
        2249,
}


assert INPUT_DIM == (
    expected_dims[
        FEATURE_MODE
    ]
), (
    FEATURE_MODE,
    INPUT_DIM,
)


print(
    "Input feature shape:",
    x_np.shape,
)

print(
    "Input dim:",
    INPUT_DIM,
)

print(
    "Feature SHA256:",
    sha256_file(
        FEATURE_PATH
    ),
)


# ============================================================
# 10) LOAD ORIGINAL SPLITS
# ============================================================

frames = {

    "train":
        pd.read_csv(
            TRAIN_PATH,
            keep_default_na=False,
        ),

    "calibration":
        pd.read_csv(
            CAL_PATH,
            keep_default_na=False,
        ),

    "test":
        pd.read_csv(
            TEST_PATH,
            keep_default_na=False,
        ),
}


expected_rows = {

    "train":
        134249,

    "calibration":
        28768,

    "test":
        28768,
}


for name, frame in frames.items():

    assert len(
        frame
    ) == expected_rows[name], (
        name,
        len(frame),
    )

    assert {
        "Drug1",
        "Drug2",
        "Label",
        "label_id",
    }.issubset(
        frame.columns
    )


print("\nSplit rows:")

for name in [
    "train",
    "calibration",
    "test",
]:

    print(
        f"{name:12s}: "
        f"{len(frames[name]):,}"
    )


# ============================================================
# 11) VERIFY IMMUTABLE INTERACTION MAPPING
# ============================================================

mapping_parts = []


for name, frame in frames.items():

    mapping_parts.append(
        frame[
            [
                "label_id",
                "Label",
            ]
        ]
        .drop_duplicates()
    )


mapping = (
    pd.concat(
        mapping_parts,
        ignore_index=True,
    )
    .drop_duplicates()
)


mapping[
    "label_id"
] = mapping[
    "label_id"
].astype(int)


mapping_counts = (
    mapping
    .groupby(
        "label_id"
    )[
        "Label"
    ]
    .nunique()
)


assert mapping_counts.eq(
    1
).all()


assert set(
    mapping[
        "label_id"
    ].tolist()
) == set(
    range(84)
)


mapping = (
    mapping
    .sort_values(
        "label_id"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    mapping
) == 84


mapping.to_csv(
    OUT
    /
    "interaction_mapping_84_exact.csv",
    index=False,
)


print(
    " Exact 84 Interaction IDs/labels verified.",
    flush=True,
)


# ============================================================
# 12) MAP SPLIT DRUGS TO FEATURE ROWS
# ============================================================

drug_to_idx = {

    drug: index

    for index, drug
    in enumerate(
        all_drugs
    )
}


pair_arrays = {}
labels = {}


for name, frame in frames.items():

    labels[
        name
    ] = (
        frame[
            "label_id"
        ]
        .astype(int)
        .to_numpy(
            dtype=np.int64
        )
    )


    assert (
        labels[name].min()
        >=
        0
    )

    assert (
        labels[name].max()
        <
        84
    )


    d1 = (
        frame[
            "Drug1"
        ]
        .astype(str)
        .map(
            drug_to_idx
        )
    )


    d2 = (
        frame[
            "Drug2"
        ]
        .astype(str)
        .map(
            drug_to_idx
        )
    )


    assert not d1.isna().any(), (
        f"{name}: Drug1 not found in cache."
    )

    assert not d2.isna().any(), (
        f"{name}: Drug2 not found in cache."
    )


    pair_arrays[
        name
    ] = np.column_stack(
        [
            d1.to_numpy(
                dtype=np.int64
            ),

            d2.to_numpy(
                dtype=np.int64
            ),
        ]
    )


    assert pair_arrays[
        name
    ].shape == (
        expected_rows[name],
        2,
    )


print(
    " Drug1→Drug2 ordered pair mapping verified.",
    flush=True,
)


# ============================================================
# 13) TRAIN DRUG SCOPE
# ============================================================

train_ids = np.unique(
    pair_arrays[
        "train"
    ]
)


assert len(
    train_ids
) == 1675


np.testing.assert_array_equal(
    np.sort(
        fitted_train_indices
    ),
    train_ids,
)


train_id_set = set(
    train_ids.tolist()
)


cal_id_set = set(
    pair_arrays[
        "calibration"
    ]
    .ravel()
    .tolist()
)


test_id_set = set(
    pair_arrays[
        "test"
    ]
    .ravel()
    .tolist()
)


cal_only_ids = (
    cal_id_set
    -
    train_id_set
)


test_only_ids = (
    test_id_set
    -
    train_id_set
)


assert len(
    cal_only_ids
) == 14


assert len(
    test_only_ids
) == 20


print(
    "Train drugs:",
    len(train_ids),
)

print(
    "Calibration drugs absent from Train:",
    len(cal_only_ids),
)

print(
    "Test drugs absent from Train:",
    len(test_only_ids),
)


# ============================================================
# 14) GLOBAL -> TRAIN LOCAL INDEX
# ============================================================

global_to_train = np.full(
    len(all_drugs),
    -1,
    dtype=np.int64,
)


global_to_train[
    train_ids
] = np.arange(
    len(train_ids),
    dtype=np.int64,
)


train_pairs_local = (
    global_to_train[
        pair_arrays[
            "train"
        ]
    ]
)


assert (
    train_pairs_local
    >=
    0
).all()


# ============================================================
# 15) CORRECTED TRAIN GRAPH
# ============================================================
#
# Important:
#
# Raw Train rows:
#       134,249
#
# Unique ORIGINAL ordered pairs:
#       134,122
#
# Then reverse every one:
#       134,122
#
# Final graph:
#       268,244
#
# DO NOT globally deduplicate after adding reverses.
#
# ============================================================

forward_raw = (
    pair_arrays[
        "train"
    ]
)


assert forward_raw.shape == (
    134249,
    2,
)


# ------------------------------------------------------------
# A) Deduplicate ORIGINAL ordered interactions
# ------------------------------------------------------------

forward_unique = np.unique(
    forward_raw,
    axis=0,
)


print("\n" + "=" * 90)
print("GRAPH CONSTRUCTION AUDIT")
print("=" * 90)

print(
    "Raw Train interaction rows:",
    len(
        forward_raw
    ),
)

print(
    "Unique ordered Train edges:",
    len(
        forward_unique
    ),
)


assert forward_unique.shape == (
    134122,
    2,
), (
    "Unexpected unique ordered Train edge count:",
    forward_unique.shape,
)


# ------------------------------------------------------------
# B) Add reverse for every unique ordered edge
# ------------------------------------------------------------

reverse_unique = (
    forward_unique[
        :,
        ::-1
    ]
    .copy()
)


assert reverse_unique.shape == (
    134122,
    2,
)


# ------------------------------------------------------------
# C) Diagnostic reciprocal overlap
# ------------------------------------------------------------

forward_set = set(
    map(
        tuple,
        forward_unique.tolist(),
    )
)


reverse_set = set(
    map(
        tuple,
        reverse_unique.tolist(),
    )
)


reciprocal_directed_overlap = len(
    forward_set.intersection(
        reverse_set
    )
)


print(
    "Reciprocal directed overlap:",
    reciprocal_directed_overlap,
)


assert reciprocal_directed_overlap == 122, (
    "Unexpected reciprocal directed overlap:",
    reciprocal_directed_overlap,
)


# ------------------------------------------------------------
# D) Diagnostic ONLY:
#    globally unique union would be 268,122
#
#    THIS IS NOT USED FOR TRAINING.
# ------------------------------------------------------------

global_union_unique = np.unique(
    np.vstack(
        [
            forward_unique,
            reverse_unique,
        ]
    ),
    axis=0,
)


print(
    "Globally unique bidirectional edges "
    "(diagnostic only, NOT used):",
    len(
        global_union_unique
    ),
)


assert global_union_unique.shape == (
    268122,
    2,
)


# ------------------------------------------------------------
# E) ACTUAL GRAPH:
#    forward unique + reverse unique
#    NO second deduplication
# ------------------------------------------------------------

edge_pairs_global = np.vstack(
    [
        forward_unique,
        reverse_unique,
    ]
).astype(
    np.int64,
    copy=False,
)


assert edge_pairs_global.shape == (
    268244,
    2,
), (
    "Unexpected final graph shape:",
    edge_pairs_global.shape,
)


# ============================================================
# 16) GRAPH DRUG SCOPE QA
# ============================================================

edge_drugs = set(
    edge_pairs_global
    .ravel()
    .tolist()
)


assert edge_drugs == train_id_set, (
    "Train graph drug set differs from Train drug set."
)


assert not edge_drugs.intersection(
    cal_only_ids
), (
    "Calibration-only drug entered Train graph."
)


assert not edge_drugs.intersection(
    test_only_ids
), (
    "Test-only drug entered Train graph."
)


print(
    "Actual graph edges used:",
    len(
        edge_pairs_global
    ),
)

print(
    "Train graph drugs:",
    len(
        edge_drugs
    ),
)

print(
    "Calibration-only drugs incident to graph: 0"
)

print(
    "Test-only drugs incident to graph: 0"
)


# ============================================================
# 17) TRAIN-LOCAL GRAPH
# ============================================================

edge_pairs_train = (
    global_to_train[
        edge_pairs_global
    ]
)


assert (
    edge_pairs_train
    >=
    0
).all()


train_ids_tensor = torch.as_tensor(
    train_ids,
    dtype=torch.long,
    device=device,
)


edge_global = torch.as_tensor(
    edge_pairs_global.T,
    dtype=torch.long,
    device=device,
)


edge_train = torch.as_tensor(
    edge_pairs_train.T,
    dtype=torch.long,
    device=device,
)


assert edge_global.shape == (
    2,
    268244,
)


assert edge_train.shape == (
    2,
    268244,
)


assert torch.equal(
    train_ids_tensor[
        edge_train
    ],
    edge_global,
), (
    "Train-local graph does not resolve to global graph."
)


print(
    " Corrected 268,244-edge Train graph verified.",
    flush=True,
)


# ============================================================
# 18) FEATURE / LABEL TENSORS
# ============================================================

x_all = torch.as_tensor(
    x_np,
    dtype=torch.float32,
    device=device,
)


x_train = x_all[
    train_ids_tensor
]


assert x_train.shape[0] == 1675


train_pairs_tensor = torch.as_tensor(
    train_pairs_local,
    dtype=torch.long,
    device=device,
)


y_train = torch.as_tensor(
    labels[
        "train"
    ],
    dtype=torch.long,
    device=device,
)


assert len(
    y_train
) == 134249


# ============================================================
# 19) CLASS WEIGHTS
# ============================================================

class_counts = np.bincount(
    labels[
        "train"
    ],
    minlength=84,
)


assert len(
    class_counts
) == 84


assert (
    class_counts > 0
).all()


class_weights = np.sqrt(
    class_counts.max()
    /
    (
        class_counts
        +
        1
    )
)


class_weights = np.clip(
    class_weights,
    0.5,
    3.0,
)


class_weights = (
    class_weights
    /
    class_weights.mean()
)


class_weights_tensor = torch.as_tensor(
    class_weights,
    dtype=torch.float32,
    device=device,
)


pd.DataFrame({

    "label_id":
        np.arange(
            84
        ),

    "train_count":
        class_counts,

    "class_weight":
        class_weights,

}).to_csv(
    OUT
    /
    "train_class_weights.csv",
    index=False,
)


# ============================================================
# 20) MODEL — GCN DRUG ENCODER
# ============================================================

class GCNDrugEncoder(
    nn.Module
):

    def __init__(
        self,
        in_dim,
        hidden_dim=384,
        emb_dim=192,
        dropout=0.35,
    ):

        super().__init__()


        self.conv1 = GCNConv(
            in_dim,
            hidden_dim,
            cached=False,
            add_self_loops=True,
            normalize=True,
        )


        self.bn1 = nn.BatchNorm1d(
            hidden_dim
        )


        self.conv2 = GCNConv(
            hidden_dim,
            hidden_dim,
            cached=False,
            add_self_loops=True,
            normalize=True,
        )


        self.bn2 = nn.BatchNorm1d(
            hidden_dim
        )


        self.conv3 = GCNConv(
            hidden_dim,
            emb_dim,
            cached=False,
            add_self_loops=True,
            normalize=True,
        )


        self.bn3 = nn.BatchNorm1d(
            emb_dim
        )


        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        edge_index,
    ):

        x = self.conv1(
            x,
            edge_index,
        )

        x = self.bn1(
            x
        )

        x = torch.relu(
            x
        )

        x = self.dropout(
            x
        )


        x = self.conv2(
            x,
            edge_index,
        )

        x = self.bn2(
            x
        )

        x = torch.relu(
            x
        )

        x = self.dropout(
            x
        )


        x = self.conv3(
            x,
            edge_index,
        )

        x = self.bn3(
            x
        )


        # Deliberately no final activation
        return x


# ============================================================
# 21) FINAL ORDERED-CONCAT MODEL
# ============================================================

class FinalConcatOnlyGCN(
    nn.Module
):

    def __init__(
        self,
        in_dim,
        num_classes=84,
        dropout=0.35,
    ):

        super().__init__()


        self.encoder = GCNDrugEncoder(
            in_dim=in_dim,
            hidden_dim=384,
            emb_dim=192,
            dropout=dropout,
        )


        # Drug1 192 || Drug2 192 = 384
        self.classifier = nn.Sequential(

            nn.LayerNorm(
                384
            ),

            nn.Linear(
                384,
                512,
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                512,
                256,
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                256,
                num_classes,
            ),
        )


    def classify(
        self,
        z,
        pairs,
    ):

        z1 = z[
            pairs[
                :,
                0
            ]
        ]


        z2 = z[
            pairs[
                :,
                1
            ]
        ]


        ordered_pair = torch.cat(
            [
                z1,
                z2,
            ],
            dim=1,
        )


        assert ordered_pair.shape[
            1
        ] == 384


        return self.classifier(
            ordered_pair
        )


# ============================================================
# 22) REPRODUCIBILITY / SEED
# ============================================================

random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)

torch.cuda.manual_seed_all(
    SEED
)


torch.backends.cudnn.benchmark = False

torch.backends.cudnn.deterministic = True


# ============================================================
# 23) MODEL
# ============================================================

model = FinalConcatOnlyGCN(
    in_dim=INPUT_DIM,
).to(
    device
)


total_parameters = sum(
    p.numel()
    for p in model.parameters()
)


trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("\n" + "=" * 90)
print("MODEL")
print("=" * 90)

print(
    "Feature mode:",
    FEATURE_MODE,
)

print(
    "Input dimension:",
    INPUT_DIM,
)

print(
    "Total parameters:",
    total_parameters,
)

print(
    "Trainable parameters:",
    trainable_parameters,
)


# ============================================================
# 24) OPTIMIZER / SCHEDULER / LOSS
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=CONFIG[
        "learning_rate"
    ],

    weight_decay=CONFIG[
        "weight_decay"
    ],
)


scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(

        optimizer,

        mode="max",

        factor=CONFIG[
            "scheduler_factor"
        ],

        patience=CONFIG[
            "scheduler_patience"
        ],
    )
)


criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ============================================================
# 25) RUN IDENTITY
# ============================================================

identity = {

    "config":
        CONFIG,

    "input_dim":
        INPUT_DIM,

    "total_parameters":
        total_parameters,

    "trainable_parameters":
        trainable_parameters,

    "feature_sha256":
        sha256_file(
            FEATURE_PATH
        ),

    "feature_metadata_sha256":
        sha256_file(
            FEATURE_META_PATH
        ),

    "split_sha256": {

        "train":
            sha256_file(
                TRAIN_PATH
            ),

        "calibration":
            sha256_file(
                CAL_PATH
            ),

        "test":
            sha256_file(
                TEST_PATH
            ),
    },

    "torch":
        str(
            torch.__version__
        ),

    "torch_geometric":
        str(
            torch_geometric.__version__
        ),

    "gpu":
        torch.cuda.get_device_name(
            0
        ),
}


identity_path = (
    OUT
    /
    "run_identity.json"
)


if identity_path.exists():

    previous_identity = json.loads(
        identity_path.read_text(
            encoding="utf-8"
        )
    )


    assert previous_identity == identity, (
        "Saved run identity differs.\n"
        "Delete nothing and do not mix experiments."
    )


else:

    save_json(
        identity,
        identity_path,
    )


# ============================================================
# 26) EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(
    split_name,
):

    model.eval()


    # --------------------------------------------------------
    # Full molecular feature matrix is available.
    #
    # Interaction graph remains TRAIN-only.
    #
    # Non-Train drugs therefore receive no interaction
    # neighbors from calibration/test.
    # --------------------------------------------------------

    z = model.encoder(
        x_all,
        edge_global,
    )


    assert torch.isfinite(
        z
    ).all()


    predictions = []


    split_pairs = (
        pair_arrays[
            split_name
        ]
    )


    for start in range(
        0,
        len(
            split_pairs
        ),
        2048,
    ):

        pairs = torch.as_tensor(
            split_pairs[
                start:
                start
                +
                2048
            ],
            dtype=torch.long,
            device=device,
        )


        logits = model.classify(
            z,
            pairs,
        )


        assert torch.isfinite(
            logits
        ).all()


        predictions.append(

            logits
            .argmax(
                dim=1
            )
            .cpu()
            .numpy()
        )


    predictions = np.concatenate(
        predictions
    )


    y_true = (
        labels[
            split_name
        ]
    )


    observed_classes = np.unique(
        y_true
    )


    metrics = {

        "accuracy":
            float(
                accuracy_score(
                    y_true,
                    predictions,
                )
            ),

        "macro_observed":
            float(
                f1_score(
                    y_true,
                    predictions,
                    labels=observed_classes,
                    average="macro",
                    zero_division=0,
                )
            ),

        "macro84":
            float(
                f1_score(
                    y_true,
                    predictions,
                    labels=np.arange(
                        84
                    ),
                    average="macro",
                    zero_division=0,
                )
            ),

        "weighted_f1":
            float(
                f1_score(
                    y_true,
                    predictions,
                    average="weighted",
                    zero_division=0,
                )
            ),
    }


    return (
        metrics,
        predictions,
    )


# ============================================================
# 27) CHECKPOINT PATHS
# ============================================================

last_path = (
    OUT
    /
    "last_checkpoint.pt"
)


best_path = (
    OUT
    /
    "best_checkpoint.pt"
)


history_path = (
    OUT
    /
    "history.csv"
)


result_path = (
    OUT
    /
    "final_result.json"
)


# ============================================================
# 28) INITIAL TRAINING STATE
# ============================================================

start_epoch = 1

best_epoch = 0

best_score = -np.inf

best_model = None

wait = 0

history = []


# ============================================================
# 29) TRAIN / AUTO-RESUME
# ============================================================

if not result_path.exists():

    # --------------------------------------------------------
    # Resume if interrupted previously
    # --------------------------------------------------------

    if last_path.exists():

        checkpoint = torch.load(
            last_path,
            map_location="cpu",
            weights_only=False,
        )


        assert (
            checkpoint[
                "identity"
            ]
            ==
            identity
        )


        model.load_state_dict(
            checkpoint[
                "model"
            ]
        )


        optimizer.load_state_dict(
            checkpoint[
                "optimizer"
            ]
        )


        scheduler.load_state_dict(
            checkpoint[
                "scheduler"
            ]
        )


        start_epoch = (
            checkpoint[
                "epoch"
            ]
            +
            1
        )


        best_epoch = checkpoint[
            "best_epoch"
        ]


        best_score = checkpoint[
            "best_score"
        ]


        best_model = checkpoint[
            "best_model"
        ]


        wait = checkpoint[
            "wait"
        ]


        history = checkpoint[
            "history"
        ]


        torch.set_rng_state(
            checkpoint[
                "torch_rng"
            ]
        )


        torch.cuda.set_rng_state_all(
            checkpoint[
                "cuda_rng"
            ]
        )


        np.random.set_state(
            checkpoint[
                "numpy_rng"
            ]
        )


        random.setstate(
            checkpoint[
                "python_rng"
            ]
        )


        print(
            "\nResuming after epoch:",
            start_epoch - 1,
            flush=True,
        )


    print("\n" + "=" * 90)
    print("TRAINING")
    print("=" * 90)

    print(
        "Feature mode:",
        FEATURE_MODE,
    )

    print(
        "Seed:",
        SEED,
    )

    print(
        "Max epochs:",
        CONFIG[
            "max_epochs"
        ],
    )

    print(
        "Early stopping patience:",
        CONFIG[
            "early_stopping_patience"
        ],
    )

    print(
        "Selection:",
        CONFIG[
            "model_selection"
        ],
    )


    try:

        for epoch in range(
            start_epoch,
            CONFIG[
                "max_epochs"
            ]
            +
            1,
        ):

            # ------------------------------------------------
            # Early stopping
            # ------------------------------------------------

            if wait >= CONFIG[
                "early_stopping_patience"
            ]:

                print(
                    "Early stopping reached.",
                    flush=True,
                )

                break


            model.train()


            order = torch.randperm(
                len(
                    y_train
                ),
                device=device,
            )


            loss_total = 0.0


            # ------------------------------------------------
            # Pair minibatches
            #
            # Drug encoder is recomputed on Train drugs.
            # ------------------------------------------------

            for batch_number, start in enumerate(

                range(
                    0,
                    len(
                        y_train
                    ),
                    CONFIG[
                        "batch_size"
                    ],
                ),

                start=1,
            ):

                indices = order[
                    start:
                    start
                    +
                    CONFIG[
                        "batch_size"
                    ]
                ]


                optimizer.zero_grad(
                    set_to_none=True
                )


                # --------------------------------------------
                # Train drugs ONLY
                # Train interaction graph ONLY
                # --------------------------------------------

                z = model.encoder(
                    x_train,
                    edge_train,
                )


                logits = model.classify(
                    z,
                    train_pairs_tensor[
                        indices
                    ],
                )


                loss = criterion(
                    logits,
                    y_train[
                        indices
                    ],
                )


                assert torch.isfinite(
                    loss
                ), (
                    "Non-finite training loss."
                )


                loss.backward()


                optimizer.step()


                loss_total += (
                    loss.item()
                    *
                    len(
                        indices
                    )
                )


                if (
                    batch_number == 1
                    or
                    batch_number % 80 == 0
                ):

                    print(
                        f"{FEATURE_MODE} | "
                        f"Seed {SEED} | "
                        f"Epoch {epoch:03d} | "
                        f"Train "
                        f"{min(start + len(indices), len(y_train)):,}"
                        f"/{len(y_train):,} | "
                        f"Loss={loss.item():.5f}",
                        flush=True,
                    )


            # =================================================
            # CALIBRATION EVALUATION
            # =================================================

            cal_metrics, _ = evaluate(
                "calibration"
            )


            score = (
                cal_metrics[
                    "macro84"
                ]
            )


            # =================================================
            # MODEL SELECTION — CALIBRATION ONLY
            # =================================================

            if score > best_score:

                best_score = score

                best_epoch = epoch

                wait = 0

                best_model = cpu_state(
                    model
                )


            else:

                wait += 1


            scheduler.step(
                score
            )


            epoch_record = {

                "feature_mode":
                    FEATURE_MODE,

                "seed":
                    SEED,

                "epoch":
                    epoch,

                "train_loss":
                    (
                        loss_total
                        /
                        len(
                            y_train
                        )
                    ),

                "cal_accuracy":
                    cal_metrics[
                        "accuracy"
                    ],

                "cal_macro_observed":
                    cal_metrics[
                        "macro_observed"
                    ],

                "cal_macro84":
                    cal_metrics[
                        "macro84"
                    ],

                "cal_weighted_f1":
                    cal_metrics[
                        "weighted_f1"
                    ],

                "best_cal_macro84":
                    best_score,

                "best_epoch":
                    best_epoch,

                "wait":
                    wait,

                "lr":
                    optimizer
                    .param_groups[
                        0
                    ][
                        "lr"
                    ],
            }


            history.append(
                epoch_record
            )


            # =================================================
            # SAVE AFTER EVERY COMPLETED EPOCH
            # =================================================

            save_checkpoint(

                {

                    "identity":
                        identity,

                    "epoch":
                        epoch,

                    "model":
                        cpu_state(
                            model
                        ),

                    "optimizer":
                        optimizer.state_dict(),

                    "scheduler":
                        scheduler.state_dict(),

                    "best_model":
                        best_model,

                    "best_score":
                        best_score,

                    "best_epoch":
                        best_epoch,

                    "wait":
                        wait,

                    "history":
                        history,

                    "torch_rng":
                        torch.get_rng_state(),

                    "cuda_rng":
                        torch.cuda.get_rng_state_all(),

                    "numpy_rng":
                        np.random.get_state(),

                    "python_rng":
                        random.getstate(),
                },

                last_path,
            )


            pd.DataFrame(
                history
            ).to_csv(
                history_path,
                index=False,
            )


            print(
                f" Epoch {epoch} saved | "
                f"CAL Macro84={score:.6f} | "
                f"Best={best_score:.6f} | "
                f"Best Epoch={best_epoch} | "
                f"Wait={wait}/"
                f"{CONFIG['early_stopping_patience']}",
                flush=True,
            )


    except KeyboardInterrupt:

        print(
            "\nTraining interrupted safely.\n"
            "Rerun this SAME cell with the SAME "
            "FEATURE_MODE and SEED to resume "
            "from the last completed epoch.",
            flush=True,
        )

        raise


    # ========================================================
    # 30) SAVE BEST CHECKPOINT
    # ========================================================

    assert best_model is not None


    save_checkpoint(

        {

            "identity":
                identity,

            "model":
                best_model,

            "best_epoch":
                best_epoch,

            "cal_macro84":
                best_score,
        },

        best_path,
    )


    # ========================================================
    # 31) RELOAD BEST CHECKPOINT
    # ========================================================

    restored = torch.load(
        best_path,
        map_location="cpu",
        weights_only=False,
    )


    assert (
        restored[
            "identity"
        ]
        ==
        identity
    )


    model.load_state_dict(
        restored[
            "model"
        ]
    )


    # ========================================================
    # 32) VERIFY BEST CALIBRATION SCORE
    # ========================================================

    cal_check, _ = evaluate(
        "calibration"
    )


    checkpoint_gap = abs(
        cal_check[
            "macro84"
        ]
        -
        best_score
    )


    print(
        "\nBest epoch:",
        best_epoch,
    )

    print(
        "Saved best CAL Macro84:",
        best_score,
    )

    print(
        "Reloaded CAL Macro84:",
        cal_check[
            "macro84"
        ],
    )

    print(
        "Checkpoint gap:",
        checkpoint_gap,
    )


    assert checkpoint_gap <= 1e-12, (
        "Best-checkpoint mismatch: "
        f"{checkpoint_gap}"
    )


    # ========================================================
    # 33) TEST — ONLY AFTER MODEL SELECTION
    # ========================================================

    print("\n" + "=" * 90)
    print("FINAL TEST EVALUATION")
    print("=" * 90)

    print(
        "Evaluating TEST only after "
        "Calibration-based checkpoint selection...",
        flush=True,
    )


    test_metrics, predictions = evaluate(
        "test"
    )


    # ========================================================
    # 34) SAVE TEST PREDICTIONS
    # ========================================================

    prediction_df = (
        frames[
            "test"
        ]
        .copy()
    )


    prediction_df[
        "pred_label_id"
    ] = predictions


    prediction_df[
        "correct"
    ] = (
        predictions
        ==
        labels[
            "test"
        ]
    )


    prediction_path = (
        OUT
        /
        "test_predictions.csv"
    )


    prediction_df.to_csv(
        prediction_path,
        index=False,
    )


    # Verify written predictions
    checked_predictions = pd.read_csv(
        prediction_path
    )


    np.testing.assert_array_equal(

        checked_predictions[
            "pred_label_id"
        ].to_numpy(),

        predictions,
    )


    # ========================================================
    # 35) FINAL RESULT
    # ========================================================

    result = {

        "experiment":
            "controlled_feature_ablation_train_only",

        "feature_mode":
            FEATURE_MODE,

        "seed":
            SEED,

        "input_dim":
            INPUT_DIM,

        "model":
            "concat_only_gcn",

        "graph":
            "TRAIN_ORDERED_UNIQUE_THEN_REVERSE",

        "graph_forward_unique_edges":
            134122,

        "graph_total_edges":
            268244,

        "graph_reciprocal_overlap":
            reciprocal_directed_overlap,

        "train_rows":
            134249,

        "calibration_rows":
            28768,

        "test_rows":
            28768,

        "train_fit_drugs":
            1675,

        "total_drugs":
            1705,

        "calibration_unseen_drugs":
            14,

        "test_unseen_drugs":
            20,

        "completed_epochs":
            history[
                -1
            ][
                "epoch"
            ],

        "best_epoch":
            best_epoch,

        "cal_accuracy":
            cal_check[
                "accuracy"
            ],

        "cal_macro_observed":
            cal_check[
                "macro_observed"
            ],

        "cal_macro84":
            best_score,

        "cal_weighted_f1":
            cal_check[
                "weighted_f1"
            ],

        "checkpoint_gap":
            checkpoint_gap,

        "test_observed_classes":
            int(
                len(
                    np.unique(
                        labels[
                            "test"
                        ]
                    )
                )
            ),

        "test_accuracy":
            test_metrics[
                "accuracy"
            ],

        "test_macro_observed":
            test_metrics[
                "macro_observed"
            ],

        "test_macro84":
            test_metrics[
                "macro84"
            ],

        "test_weighted_f1":
            test_metrics[
                "weighted_f1"
            ],

        "total_parameters":
            total_parameters,

        "trainable_parameters":
            trainable_parameters,

        "best_checkpoint_sha256":
            sha256_file(
                best_path
            ),

        "test_predictions_sha256":
            sha256_file(
                prediction_path
            ),

        "feature_cache_sha256":
            sha256_file(
                FEATURE_PATH
            ),

        "feature_metadata_sha256":
            sha256_file(
                FEATURE_META_PATH
            ),
    }


    # ========================================================
    # 36) SAVE FINAL RESULT
    # ========================================================

    save_json(
        result,
        result_path,
    )


    pd.DataFrame(
        [
            result
        ]
    ).to_csv(
        OUT
        /
        "final_result.csv",
        index=False,
    )


else:

    # ========================================================
    # ALREADY COMPLETE — VERIFY INSTEAD OF RETRAIN
    # ========================================================

    result = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    assert (
        result[
            "feature_mode"
        ]
        ==
        FEATURE_MODE
    )


    assert (
        result[
            "seed"
        ]
        ==
        SEED
    )


    assert (
        result[
            "input_dim"
        ]
        ==
        INPUT_DIM
    )


    assert (
        result[
            "graph_total_edges"
        ]
        ==
        268244
    )


    assert (
        sha256_file(
            best_path
        )
        ==
        result[
            "best_checkpoint_sha256"
        ]
    )


    assert (
        sha256_file(
            OUT
            /
            "test_predictions.csv"
        )
        ==
        result[
            "test_predictions_sha256"
        ]
    )


    assert (
        sha256_file(
            FEATURE_PATH
        )
        ==
        result[
            "feature_cache_sha256"
        ]
    )


    print(
        "\n Completed result already exists."
    )

    print(
        " Hash verification passed."
    )

    print(
        " Training skipped."
    )


# ============================================================
# 37) CREATE RUN ZIP
# ============================================================

zip_path = (
    OUT.parent
    /
    (
        "FEATURE_ABLATION_"
        +
        FEATURE_MODE.upper()
        +
        "_SEED_"
        +
        str(SEED)
        +
        "_COMPLETE.zip"
    )
)


temporary_zip = zip_path.with_suffix(
    ".zip.tmp"
)


with zipfile.ZipFile(

    temporary_zip,

    "w",

    compression=zipfile.ZIP_DEFLATED,

) as archive:

    for path in sorted(
        OUT.rglob("*")
    ):

        if (
            path.is_file()
            and
            not path.name.endswith(
                ".tmp"
            )
        ):

            archive.write(

                path,

                arcname=str(
                    path.relative_to(
                        OUT.parent
                    )
                ),
            )


with zipfile.ZipFile(
    temporary_zip,
    "r",
) as archive:

    assert archive.testzip() is None


os.replace(
    temporary_zip,
    zip_path,
)


# ============================================================
# 38) FINAL REPORT
# ============================================================

print("\n" + "=" * 90)

print(
    f" FEATURE ABLATION — "
    f"{FEATURE_MODE.upper()} — "
    f"SEED {SEED} COMPLETED"
)

print("=" * 90)


print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False,
    )
)


print("\nZIP:")
print(
    zip_path
)


print("\nQA:")
print(
    " Train-only feature preprocessing"
)

print(
    " 84 Interaction IDs/labels unchanged"
)

print(
    " Ordered Drug1→Drug2 pairs preserved"
)

print(
    " Unique original Train edges = 134,122"
)

print(
    " Reverse edges added"
)

print(
    " No second global deduplication"
)

print(
    " Final Train graph edges = 268,244"
)

print(
    " Calibration-only checkpoint selection"
)

print(
    " Test evaluated only after model selection"
)

print(
    " Auto-resume enabled"
)

print(
    " ZIP integrity passed"
)

Source cell: 74.


In [ ]:
# ============================================================
# MORGAN-ONLY FEATURE ABLATION
# RUN SEEDS 43 + 44 SEQUENTIALLY
#
# Reuses the EXACT successfully executed Seed-42 trainer
# from this notebook history.
#
# Changes ONLY:
#     SEED = 42 -> 43 -> 44
#
# Keeps EXACTLY:
# - FEATURE_MODE = "morgan_only"
# - Train-only feature cache
# - 84-class mapping
# - 134,122 unique ordered Train edges
# - + reverse edges
# - 268,244 final graph edges
# - reciprocal duplicates retained
# - same GCN
# - same ordered concat
# - same class weights
# - same optimizer / LR / scheduler
# - same max epochs / patience
# - same Calibration Macro84 checkpoint selection
# - Test only after model selection
# - auto-resume
#
# NO Seed-42 retraining.
# ============================================================

import ast
import gc
import re
import json
from pathlib import Path

import torch


# ============================================================
# 1) GET NOTEBOOK HISTORY
# ============================================================

shell = get_ipython()

assert shell is not None, (
    "Run this inside the SAME Kaggle notebook/session "
    "where Morgan-only Seed 42 was executed."
)

history_cells = list(
    shell.history_manager.input_hist_raw
)


# ============================================================
# 2) FIND THE REAL SUCCESSFUL FEATURE-ABLATION TRAINER
# ============================================================
#
# We identify it using REAL AST class definitions.
# Strings inside this wrapper DO NOT count.
#
# ============================================================

candidates = []


for history_index, source in enumerate(history_cells):

    if not isinstance(source, str):
        continue

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue


    # --------------------------------------------------------
    # Real top-level classes
    # --------------------------------------------------------

    class_names = {
        node.name
        for node in tree.body
        if isinstance(node, ast.ClassDef)
    }


    if "GCNDrugEncoder" not in class_names:
        continue

    if "FinalConcatOnlyGCN" not in class_names:
        continue


    # --------------------------------------------------------
    # Find REAL top-level FEATURE_MODE / SEED assignments
    # --------------------------------------------------------

    feature_mode_value = None
    seed_value = None


    for node in tree.body:

        if not isinstance(node, ast.Assign):
            continue

        if len(node.targets) != 1:
            continue

        target = node.targets[0]

        if not isinstance(target, ast.Name):
            continue


        if target.id == "FEATURE_MODE":

            if (
                isinstance(node.value, ast.Constant)
                and
                isinstance(node.value.value, str)
            ):
                feature_mode_value = node.value.value


        if target.id == "SEED":

            if (
                isinstance(node.value, ast.Constant)
                and
                isinstance(node.value.value, int)
            ):
                seed_value = node.value.value


    # --------------------------------------------------------
    # We need the successfully used Morgan Seed-42 trainer
    # --------------------------------------------------------

    if feature_mode_value != "morgan_only":
        continue

    if seed_value != 42:
        continue


    required_terms = [

        "FEATURE_ABLATION_CONTROLLED_V1",

        "TRAIN_ORDERED_UNIQUE_THEN_REVERSE",

        "268244",

        "134122",

        "Calibration Macro-F1 over 84 classes",

        "FinalConcatOnlyGCN",

        "best_checkpoint.pt",

        "test_predictions.csv",

        "final_result.json",

        "FEATURE ABLATION",

        "Auto-resume enabled",
    ]


    if not all(
        term in source
        for term in required_terms
    ):
        continue


    candidates.append(
        {
            "history_index":
                history_index,

            "source":
                source,

            "tree":
                tree,
        }
    )


assert candidates, (
    "Original Morgan-only seed-42 source not found "
    "in the current session history.\n\n"
    "Do not rerun completed seed 42.\n"
    "Use the Kaggle session containing the source training cell."
)


selected = candidates[-1]

source42 = selected["source"]


print("=" * 90)
print("REFERENCE TRAINER RECOVERED")
print("=" * 90)

print(
    "History index:",
    selected["history_index"],
)

print(
    "Source length:",
    len(source42),
)

print(
    " Morgan-only Seed 42 trainer recovered."
)


# ============================================================
# 3) STRICT REFERENCE QA
# ============================================================

ast.parse(
    source42
)


assert 'FEATURE_MODE = "morgan_only"' in source42

assert re.search(
    r"(?m)^\s*SEED\s*=\s*42\s*$",
    source42,
), (
    "Could not uniquely identify SEED = 42."
)


assert (
    "TRAIN_ORDERED_UNIQUE_THEN_REVERSE"
    in source42
)

assert (
    "graph_forward_unique_edges"
    in source42
)

assert "134122" in source42
assert "268244" in source42

assert (
    "Calibration Macro-F1 over 84 classes"
    in source42
)

assert (
    "test_predictions.csv"
    in source42
)

assert (
    "best_checkpoint.pt"
    in source42
)


print(
    " Seed-42 reference trainer passed protocol QA."
)


# ============================================================
# 4) FUNCTION TO GENERATE A NEW SEED SCRIPT
# ============================================================

def build_seed_source(
    original_source,
    new_seed,
):

    assert new_seed in {
        43,
        44,
    }


    # --------------------------------------------------------
    # Replace ONLY the real line:
    #
    #     SEED = 42
    #
    # Nothing else.
    # --------------------------------------------------------

    pattern = (
        r"(?m)^(\s*)SEED\s*=\s*42\s*$"
    )


    matches = re.findall(
        pattern,
        original_source,
    )


    assert len(matches) == 1, (
        "Expected exactly one real SEED = 42 assignment, "
        f"found {len(matches)}."
    )


    new_source = re.sub(
        pattern,
        lambda match:
            (
                match.group(1)
                +
                f"SEED = {new_seed}"
            ),
        original_source,
        count=1,
    )


    # --------------------------------------------------------
    # Syntax check
    # --------------------------------------------------------

    tree = ast.parse(
        new_source
    )


    # --------------------------------------------------------
    # Verify actual AST assignment
    # --------------------------------------------------------

    found_seed = None
    found_mode = None


    for node in tree.body:

        if not isinstance(
            node,
            ast.Assign,
        ):
            continue

        if len(
            node.targets
        ) != 1:
            continue

        target = node.targets[0]

        if not isinstance(
            target,
            ast.Name,
        ):
            continue


        if target.id == "SEED":

            if isinstance(
                node.value,
                ast.Constant,
            ):
                found_seed = (
                    node.value.value
                )


        if target.id == "FEATURE_MODE":

            if isinstance(
                node.value,
                ast.Constant,
            ):
                found_mode = (
                    node.value.value
                )


    assert found_seed == new_seed

    assert (
        found_mode
        ==
        "morgan_only"
    )


    # --------------------------------------------------------
    # Protocol must remain unchanged
    # --------------------------------------------------------

    critical_terms = [

        "TRAIN_ORDERED_UNIQUE_THEN_REVERSE",

        "134122",

        "268244",

        "Calibration Macro-F1 over 84 classes",

        "ordered_concat_z1_z2",

        "weighted CrossEntropyLoss",

        "early_stopping_patience",

        "test_predictions.csv",

        "best_checkpoint.pt",
    ]


    for term in critical_terms:

        assert term in new_source, (
            f"Protocol term disappeared: {term}"
        )


    return new_source


# ============================================================
# 5) GENERATE SEED 43 + 44 SOURCES
# ============================================================

source43 = build_seed_source(
    source42,
    43,
)


source44 = build_seed_source(
    source42,
    44,
)


print(
    " Seed 43 source generated."
)

print(
    " Seed 44 source generated."
)


# ============================================================
# 6) SAVE EXACT GENERATED TRAINERS
# ============================================================

RUN_ROOT = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_CONTROLLED_V1"
    "/morgan_only"
)


RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


generated_scripts = {}


for seed, source in [
    (43, source43),
    (44, source44),
]:

    seed_dir = (
        RUN_ROOT
        /
        f"seed_{seed}"
    )

    seed_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    script_path = (
        seed_dir
        /
        f"train_morgan_only_seed_{seed}.py"
    )


    if script_path.exists():

        existing = script_path.read_text(
            encoding="utf-8"
        )

        assert existing == source, (
            f"A different Seed {seed} script "
            "already exists.\n"
            "Stopping rather than mixing experiment definitions."
        )

    else:

        script_path.write_text(
            source,
            encoding="utf-8",
        )


    generated_scripts[
        seed
    ] = script_path


    print(
        f" Seed {seed} trainer saved:",
        script_path,
    )


# ============================================================
# 7) PRESERVE SEED-42 REFERENCE SOURCE
# ============================================================

reference_path = (
    RUN_ROOT
    /
    "reference_successful_morgan_seed42_trainer.txt"
)


if reference_path.exists():

    assert (
        reference_path.read_text(
            encoding="utf-8"
        )
        ==
        source42
    )

else:

    reference_path.write_text(
        source42,
        encoding="utf-8",
    )


print(
    " Exact successful Seed-42 trainer preserved."
)


# ============================================================
# 8) RUN HELPER
# ============================================================

def run_seed(
    seed,
    source,
    script_path,
):

    print(
        "\n"
        +
        "=" * 90
    )

    print(
        f"STARTING MORGAN-ONLY — SEED {seed}"
    )

    print(
        "=" * 90
    )

    print(
        " Same Train-only Morgan cache"
    )

    print(
        " Same original Train/Calibration/Test splits"
    )

    print(
        " Same 84 Interaction classes"
    )

    print(
        " Same ordered Drug1→Drug2 representation"
    )

    print(
        " Same 134,122 forward unique Train edges"
    )

    print(
        " Same 268,244 final graph edges"
    )

    print(
        " Same model / loss / optimizer / scheduler"
    )

    print(
        " Same Calibration Macro84 selection"
    )

    print(
        " Test only after model selection"
    )

    print(
        " Auto-resume enabled"
    )

    print(
        f"\nRunning Seed {seed}...",
        flush=True,
    )


    namespace = {

        "__name__":
            "__main__",

        "__file__":
            str(
                script_path
            ),
    }


    exec(
        compile(
            source,
            str(
                script_path
            ),
            "exec",
        ),
        namespace,
    )


    # --------------------------------------------------------
    # Confirm final result exists
    # --------------------------------------------------------

    result_path = (
        RUN_ROOT
        /
        f"seed_{seed}"
        /
        "final_result.json"
    )


    assert result_path.is_file(), (
        f"Seed {seed} finished without final_result.json."
    )


    result = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    assert result[
        "feature_mode"
    ] == "morgan_only"

    assert result[
        "seed"
    ] == seed

    assert result[
        "graph_total_edges"
    ] == 268244


    print(
        "\n"
        +
        "-" * 90
    )

    print(
        f" MORGAN-ONLY SEED {seed} VERIFIED"
    )

    print(
        "-" * 90
    )

    print(
        "Best epoch:",
        result[
            "best_epoch"
        ],
    )

    print(
        "CAL Macro84:",
        result[
            "cal_macro84"
        ],
    )

    print(
        "TEST Accuracy:",
        result[
            "test_accuracy"
        ],
    )

    print(
        "TEST Macro84:",
        result[
            "test_macro84"
        ],
    )

    print(
        "TEST Weighted-F1:",
        result[
            "test_weighted_f1"
        ],
    )


    # --------------------------------------------------------
    # Free temporary GPU memory between seeds
    # --------------------------------------------------------

    del namespace

    gc.collect()

    torch.cuda.empty_cache()


    return result


# ============================================================
# 9) RUN SEED 43
# ============================================================

result43 = run_seed(

    seed=43,

    source=source43,

    script_path=generated_scripts[
        43
    ],
)


# ============================================================
# 10) RUN SEED 44
# ============================================================

result44 = run_seed(

    seed=44,

    source=source44,

    script_path=generated_scripts[
        44
    ],
)


# ============================================================
# 11) FINAL TWO-SEED SUMMARY
# ============================================================

summary = []


for result in [
    result43,
    result44,
]:

    summary.append({

        "seed":
            result[
                "seed"
            ],

        "best_epoch":
            result[
                "best_epoch"
            ],

        "completed_epochs":
            result[
                "completed_epochs"
            ],

        "cal_macro84":
            result[
                "cal_macro84"
            ],

        "test_accuracy":
            result[
                "test_accuracy"
            ],

        "test_macro84":
            result[
                "test_macro84"
            ],

        "test_weighted_f1":
            result[
                "test_weighted_f1"
            ],
    })


import pandas as pd


summary_df = pd.DataFrame(
    summary
)


summary_path = (
    RUN_ROOT
    /
    "morgan_only_seeds_43_44_summary.csv"
)


summary_df.to_csv(
    summary_path,
    index=False,
)


print(
    "\n"
    +
    "=" * 90
)

print(
    " MORGAN-ONLY SEEDS 43 + 44 COMPLETE"
)

print(
    "=" * 90
)


display(
    summary_df
)


print(
    "\nSeed 43 test Macro-F1:",
    f"{result43['test_macro84'] * 100:.4f}%"
)

print(
    "Seed 44 test Macro-F1:",
    f"{result44['test_macro84'] * 100:.4f}%"
)


print(
    "\n Seed 42 was NOT retrained."
)

print(
    " Seeds 43 and 44 use the exact successful Seed-42 trainer."
)

print(
    " Only the random seed changed."
)

print(
    " Results remain in separate seed directories."
)

print(
    "\nSummary saved:"
)

print(
    summary_path
)

Source cell: 75.


In [ ]:
# ============================================================
# RDKIT-ONLY FEATURE ABLATION
# RUN SEEDS 42 + 43 + 44 SEQUENTIALLY
# ============================================================
#
# Uses the EXACT successful controlled feature-ablation trainer
# already preserved from Morgan-only Seed 42.
#
# Changes ONLY:
#
#     FEATURE_MODE = "rdkit_only"
#     SEED = 42 / 43 / 44
#
# Everything else remains identical:
#
# - Exact original Train / Calibration / Test splits
# - 84 Interaction classes
# - Train-only RDKit preprocessing
# - RDKit input dim = 201
# - 134,122 unique ordered Train edges
# - reverse edges added
# - reciprocal overlap retained
# - 268,244 final graph edges
# - same GCN architecture
# - same ordered concat
# - same class weighting
# - same optimizer / LR / scheduler
# - same max epochs / patience
# - same Calibration Macro84 checkpoint selection
# - Test used only after model selection
# - auto-resume
#
# ============================================================


# ============================================================
# 1) IMPORTS
# ============================================================

import ast
import gc
import re
import json
import os
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

import torch

from IPython.display import display


# ============================================================
# 2) PATHS
# ============================================================

FEATURE_ROOT = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_CONTROLLED_V1"
)

MORGAN_ROOT = (
    RUN_ROOT
    /
    "morgan_only"
)

RDKIT_ROOT = (
    RUN_ROOT
    /
    "rdkit_only"
)

RDKIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


RDKIT_CACHE = (
    FEATURE_ROOT
    /
    "molecular_features_rdkit_only.npz"
)


REFERENCE_FILE = (
    MORGAN_ROOT
    /
    "reference_successful_morgan_seed42_trainer.txt"
)


# ============================================================
# 3) BASIC INPUT QA
# ============================================================

assert RDKIT_CACHE.is_file(), (
    "Missing RDKit-only feature cache:\n"
    f"{RDKIT_CACHE}\n\n"
    "Cell A should already have created it."
)


with np.load(
    RDKIT_CACHE,
    allow_pickle=True,
) as data:

    rdkit_shape = data["x"].shape

    all_drugs = (
        data["all_drugs"]
        .astype(str)
    )

    train_indices = (
        data["train_drug_indices"]
        .astype(np.int64)
    )


assert rdkit_shape == (
    1705,
    201,
), (
    "Unexpected RDKit-only feature shape:",
    rdkit_shape,
)


assert len(
    all_drugs
) == 1705


assert len(
    train_indices
) == 1675


print("=" * 90)
print("RDKIT-ONLY INPUT VERIFIED")
print("=" * 90)

print(
    "RDKit feature shape:",
    rdkit_shape,
)

print(
    "Total drugs:",
    len(all_drugs),
)

print(
    "Train-fit drugs:",
    len(train_indices),
)

print(
    " RDKit-only cache ready."
)


# ============================================================
# 4) RECOVER EXACT SUCCESSFUL TRAINER
# ============================================================
#
# Preferred source:
# reference_successful_morgan_seed42_trainer.txt
#
# If that file is absent, recover from current notebook history.
#
# ============================================================

source42 = None


if REFERENCE_FILE.is_file():

    source42 = (
        REFERENCE_FILE.read_text(
            encoding="utf-8"
        )
    )

    print(
        "\n Using preserved successful "
        "Morgan Seed-42 trainer:"
    )

    print(
        REFERENCE_FILE
    )


else:

    print(
        "\nReference trainer file not found."
    )

    print(
        "Trying current notebook history..."
    )


    shell = get_ipython()

    assert shell is not None, (
        "Run this inside the Kaggle notebook "
        "where the successful trainer was executed."
    )


    history_cells = list(
        shell.history_manager.input_hist_raw
    )


    candidates = []


    for history_index, source in enumerate(
        history_cells
    ):

        if not isinstance(
            source,
            str,
        ):
            continue


        try:

            tree = ast.parse(
                source
            )

        except SyntaxError:

            continue


        class_names = {

            node.name

            for node in tree.body

            if isinstance(
                node,
                ast.ClassDef,
            )
        }


        if (
            "GCNDrugEncoder"
            not in class_names
        ):

            continue


        if (
            "FinalConcatOnlyGCN"
            not in class_names
        ):

            continue


        feature_mode_value = None
        seed_value = None


        for node in tree.body:

            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if len(
                node.targets
            ) != 1:
                continue


            target = node.targets[0]


            if not isinstance(
                target,
                ast.Name,
            ):
                continue


            if target.id == "FEATURE_MODE":

                if (
                    isinstance(
                        node.value,
                        ast.Constant,
                    )
                    and
                    isinstance(
                        node.value.value,
                        str,
                    )
                ):

                    feature_mode_value = (
                        node.value.value
                    )


            if target.id == "SEED":

                if (
                    isinstance(
                        node.value,
                        ast.Constant,
                    )
                    and
                    isinstance(
                        node.value.value,
                        int,
                    )
                ):

                    seed_value = (
                        node.value.value
                    )


        if (
            feature_mode_value
            !=
            "morgan_only"
        ):

            continue


        if seed_value != 42:

            continue


        required_terms = [

            "FEATURE_ABLATION_CONTROLLED_V1",

            "TRAIN_ORDERED_UNIQUE_THEN_REVERSE",

            "134122",

            "268244",

            "FinalConcatOnlyGCN",

            "Calibration Macro-F1 over 84 classes",

            "best_checkpoint.pt",

            "test_predictions.csv",

            "final_result.json",
        ]


        if not all(
            term in source
            for term in required_terms
        ):

            continue


        candidates.append(
            (
                history_index,
                source,
            )
        )


    assert candidates, (
        "Could not recover the successful "
        "Morgan-only Seed-42 trainer."
    )


    history_index, source42 = (
        candidates[-1]
    )


    print(
        " Recovered trainer from history index:",
        history_index,
    )


# ============================================================
# 5) STRICT REFERENCE TRAINER QA
# ============================================================

tree42 = ast.parse(
    source42
)


top_level_classes = {

    node.name

    for node in tree42.body

    if isinstance(
        node,
        ast.ClassDef,
    )
}


assert (
    "GCNDrugEncoder"
    in top_level_classes
)

assert (
    "FinalConcatOnlyGCN"
    in top_level_classes
)


assert (
    "TRAIN_ORDERED_UNIQUE_THEN_REVERSE"
    in source42
)

assert "134122" in source42
assert "268244" in source42

assert (
    "Calibration Macro-F1 over 84 classes"
    in source42
)

assert (
    "ordered_concat_z1_z2"
    in source42
)

assert (
    "weighted CrossEntropyLoss"
    in source42
)

assert (
    "best_checkpoint.pt"
    in source42
)

assert (
    "test_predictions.csv"
    in source42
)


print(
    " Reference trainer passed protocol QA."
)


# ============================================================
# 6) HELPER — READ TOP-LEVEL CONFIG ASSIGNMENTS
# ============================================================

def get_top_level_values(
    source,
):

    tree = ast.parse(
        source
    )


    feature_mode = None
    seed = None


    for node in tree.body:

        if not isinstance(
            node,
            ast.Assign,
        ):
            continue


        if len(
            node.targets
        ) != 1:
            continue


        target = node.targets[0]


        if not isinstance(
            target,
            ast.Name,
        ):
            continue


        if (
            target.id
            ==
            "FEATURE_MODE"
        ):

            if isinstance(
                node.value,
                ast.Constant,
            ):

                feature_mode = (
                    node.value.value
                )


        if (
            target.id
            ==
            "SEED"
        ):

            if isinstance(
                node.value,
                ast.Constant,
            ):

                seed = (
                    node.value.value
                )


    return (
        feature_mode,
        seed,
    )


reference_mode, reference_seed = (
    get_top_level_values(
        source42
    )
)


assert (
    reference_mode
    ==
    "morgan_only"
)

assert reference_seed == 42


# ============================================================
# 7) BUILD RDKIT-ONLY TRAINER SOURCE
# ============================================================

def build_rdkit_source(
    original_source,
    new_seed,
):

    assert new_seed in {
        42,
        43,
        44,
    }


    # --------------------------------------------------------
    # Replace only the REAL top-level FEATURE_MODE assignment.
    # --------------------------------------------------------

    feature_pattern = (
        r'(?m)^(\s*)FEATURE_MODE\s*=\s*'
        r'["\']morgan_only["\']\s*$'
    )


    feature_matches = re.findall(
        feature_pattern,
        original_source,
    )


    assert len(
        feature_matches
    ) == 1, (
        "Expected exactly one top-level "
        "FEATURE_MODE = morgan_only assignment."
    )


    new_source = re.sub(
        feature_pattern,
        lambda m:
            (
                m.group(1)
                +
                'FEATURE_MODE = "rdkit_only"'
            ),
        original_source,
        count=1,
    )


    # --------------------------------------------------------
    # Seed 42 stays 42.
    # Seeds 43 and 44 replace top-level SEED only.
    # --------------------------------------------------------

    if new_seed != 42:

        seed_pattern = (
            r"(?m)^(\s*)SEED\s*=\s*42\s*$"
        )


        seed_matches = re.findall(
            seed_pattern,
            new_source,
        )


        assert len(
            seed_matches
        ) == 1, (
            "Expected exactly one "
            "SEED = 42 assignment."
        )


        new_source = re.sub(
            seed_pattern,
            lambda m:
                (
                    m.group(1)
                    +
                    f"SEED = {new_seed}"
                ),
            new_source,
            count=1,
        )


    # --------------------------------------------------------
    # Syntax + AST QA
    # --------------------------------------------------------

    ast.parse(
        new_source
    )


    mode_check, seed_check = (
        get_top_level_values(
            new_source
        )
    )


    assert (
        mode_check
        ==
        "rdkit_only"
    )


    assert (
        seed_check
        ==
        new_seed
    )


    # --------------------------------------------------------
    # Critical experiment definition must remain present.
    # --------------------------------------------------------

    critical_terms = [

        "TRAIN_ORDERED_UNIQUE_THEN_REVERSE",

        "134122",

        "268244",

        "FinalConcatOnlyGCN",

        "ordered_concat_z1_z2",

        "weighted CrossEntropyLoss",

        "Calibration Macro-F1 over 84 classes",

        "best_checkpoint.pt",

        "test_predictions.csv",

        "final_result.json",

        "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING",
    ]


    for term in critical_terms:

        assert term in new_source, (
            f"Critical protocol term disappeared: "
            f"{term}"
        )


    return new_source


# ============================================================
# 8) GENERATE ALL THREE RDKIT TRAINERS
# ============================================================

generated_sources = {}
generated_scripts = {}


for seed in [
    42,
    43,
    44,
]:

    source = build_rdkit_source(
        source42,
        seed,
    )


    seed_dir = (
        RDKIT_ROOT
        /
        f"seed_{seed}"
    )


    seed_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    script_path = (
        seed_dir
        /
        f"train_rdkit_only_seed_{seed}.py"
    )


    if script_path.exists():

        existing = (
            script_path.read_text(
                encoding="utf-8"
            )
        )


        assert existing == source, (
            f"\nA DIFFERENT Seed {seed} "
            "trainer already exists:\n"
            f"{script_path}\n\n"
            "Stopping rather than mixing "
            "experiment definitions."
        )


    else:

        script_path.write_text(
            source,
            encoding="utf-8",
        )


    generated_sources[
        seed
    ] = source


    generated_scripts[
        seed
    ] = script_path


    print(
        f" RDKit-only Seed {seed} "
        f"trainer prepared:"
    )

    print(
        script_path
    )


# ============================================================
# 9) RUN FUNCTION
# ============================================================

def run_one_seed(
    seed,
):

    source = (
        generated_sources[
            seed
        ]
    )


    script_path = (
        generated_scripts[
            seed
        ]
    )


    result_path = (
        RDKIT_ROOT
        /
        f"seed_{seed}"
        /
        "final_result.json"
    )


    print(
        "\n"
        +
        "=" * 90
    )

    print(
        f"RDKIT-ONLY — SEED {seed}"
    )

    print(
        "=" * 90
    )


    print(
        "Expected input dim: 201"
    )

    print(
        "Feature cache:",
        RDKIT_CACHE
    )

    print(
        " Same train/cal/test splits"
    )

    print(
        " Same 84 classes"
    )

    print(
        " Same GCN"
    )

    print(
        " Same ordered concat"
    )

    print(
        " Same class weights"
    )

    print(
        " Same 134,122 unique forward edges"
    )

    print(
        " Same 268,244 final graph edges"
    )

    print(
        " Calibration-only model selection"
    )

    print(
        " Test used only after checkpoint selection"
    )

    print(
        " Auto-resume active"
    )


    # --------------------------------------------------------
    # Execute exact generated trainer
    # --------------------------------------------------------

    namespace = {

        "__name__":
            "__main__",

        "__file__":
            str(
                script_path
            ),
    }


    exec(
        compile(
            source,
            str(
                script_path
            ),
            "exec",
        ),
        namespace,
    )


    # --------------------------------------------------------
    # Final result verification
    # --------------------------------------------------------

    assert result_path.is_file(), (
        f"Seed {seed} did not create "
        "final_result.json"
    )


    result = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    assert (
        result[
            "feature_mode"
        ]
        ==
        "rdkit_only"
    )


    assert (
        result[
            "seed"
        ]
        ==
        seed
    )


    assert (
        result[
            "input_dim"
        ]
        ==
        201
    )


    assert (
        result[
            "graph_forward_unique_edges"
        ]
        ==
        134122
    )


    assert (
        result[
            "graph_total_edges"
        ]
        ==
        268244
    )


    assert (
        result[
            "train_rows"
        ]
        ==
        134249
    )


    assert (
        result[
            "calibration_rows"
        ]
        ==
        28768
    )


    assert (
        result[
            "test_rows"
        ]
        ==
        28768
    )


    assert abs(
        result[
            "checkpoint_gap"
        ]
    ) <= 1e-12


    print(
        "\n"
        +
        "-" * 90
    )

    print(
        f" RDKIT-ONLY SEED {seed} VERIFIED"
    )

    print(
        "-" * 90
    )


    print(
        "Completed epochs:",
        result[
            "completed_epochs"
        ],
    )


    print(
        "Best epoch:",
        result[
            "best_epoch"
        ],
    )


    print(
        "CAL Macro84:",
        result[
            "cal_macro84"
        ],
    )


    print(
        "TEST Accuracy:",
        result[
            "test_accuracy"
        ],
    )


    print(
        "TEST Macro84:",
        result[
            "test_macro84"
        ],
    )


    print(
        "TEST Weighted-F1:",
        result[
            "test_weighted_f1"
        ],
    )


    # --------------------------------------------------------
    # Release Python/GPU references before next seed
    # --------------------------------------------------------

    del namespace

    gc.collect()

    torch.cuda.empty_cache()


    return result


# ============================================================
# 10) RUN ALL THREE SEEDS
# ============================================================

all_results = []


for seed in [
    42,
    43,
    44,
]:

    result = run_one_seed(
        seed
    )


    all_results.append(
        result
    )


# ============================================================
# 11) CREATE 3-SEED RESULT TABLE
# ============================================================

result_rows = []


for result in all_results:

    result_rows.append({

        "seed":
            result[
                "seed"
            ],

        "input_dim":
            result[
                "input_dim"
            ],

        "completed_epochs":
            result[
                "completed_epochs"
            ],

        "best_epoch":
            result[
                "best_epoch"
            ],

        "cal_accuracy":
            result[
                "cal_accuracy"
            ],

        "cal_macro84":
            result[
                "cal_macro84"
            ],

        "cal_weighted_f1":
            result[
                "cal_weighted_f1"
            ],

        "test_accuracy":
            result[
                "test_accuracy"
            ],

        "test_macro84":
            result[
                "test_macro84"
            ],

        "test_weighted_f1":
            result[
                "test_weighted_f1"
            ],

        "checkpoint_gap":
            result[
                "checkpoint_gap"
            ],
    })


results_df = pd.DataFrame(
    result_rows
).sort_values(
    "seed"
).reset_index(
    drop=True
)


results_csv = (
    RDKIT_ROOT
    /
    "rdkit_only_all_seed_results.csv"
)


results_df.to_csv(
    results_csv,
    index=False,
)


# ============================================================
# 12) MEAN ± SAMPLE SD
# ============================================================

metrics = [

    "cal_accuracy",

    "cal_macro84",

    "cal_weighted_f1",

    "test_accuracy",

    "test_macro84",

    "test_weighted_f1",
]


summary_record = {

    "feature_mode":
        "rdkit_only",

    "n_seeds":
        3,

    "seeds":
        "42,43,44",

    "input_dim":
        201,
}


for metric in metrics:

    values = (
        results_df[
            metric
        ]
        .to_numpy(
            dtype=float
        )
    )


    mean = float(
        values.mean()
    )


    sd = float(
        values.std(
            ddof=1
        )
    )


    summary_record[
        f"{metric}_mean"
    ] = mean


    summary_record[
        f"{metric}_sd"
    ] = sd


    summary_record[
        f"{metric}_paper"
    ] = (
        f"{mean * 100:.4f} "
        f"± "
        f"{sd * 100:.4f}%"
    )


summary_df = pd.DataFrame(
    [
        summary_record
    ]
)


summary_csv = (
    RDKIT_ROOT
    /
    "rdkit_only_mean_sd_summary.csv"
)


summary_df.to_csv(
    summary_csv,
    index=False,
)


# ============================================================
# 13) FINAL DISPLAY
# ============================================================

print(
    "\n"
    +
    "=" * 90
)

print(
    " RDKIT-ONLY — ALL 3 SEEDS COMPLETE"
)

print(
    "=" * 90
)


display(
    results_df
)


print(
    "\n"
    +
    "=" * 90
)

print(
    "MEAN ± SAMPLE SD"
)

print(
    "=" * 90
)


display(
    summary_df[
        [
            "feature_mode",

            "cal_macro84_paper",

            "test_accuracy_paper",

            "test_macro84_paper",

            "test_weighted_f1_paper",
        ]
    ]
)


# ============================================================
# 14) CREATE ONE BUNDLE ZIP FOR ALL 3 SEEDS
# ============================================================

bundle_path = Path(
    "/kaggle/working/"
    "RDKIT_ONLY_3SEEDS_42_43_44.zip"
)


temporary_bundle = (
    bundle_path.with_suffix(
        ".zip.tmp"
    )
)


if temporary_bundle.exists():

    temporary_bundle.unlink()


with zipfile.ZipFile(
    temporary_bundle,
    "w",
    compression=zipfile.ZIP_STORED,
) as archive:

    # --------------------------------------------------------
    # Add individual COMPLETE zips
    # --------------------------------------------------------

    for seed in [
        42,
        43,
        44,
    ]:

        individual_zip = (
            RDKIT_ROOT
            /
            (
                "FEATURE_ABLATION_"
                "RDKIT_ONLY_"
                f"SEED_{seed}_COMPLETE.zip"
            )
        )


        assert individual_zip.is_file(), (
            f"Missing individual ZIP:\n"
            f"{individual_zip}"
        )


        archive.write(
            individual_zip,
            arcname=(
                individual_zip.name
            ),
        )


    # --------------------------------------------------------
    # Add summary files
    # --------------------------------------------------------

    archive.write(
        results_csv,
        arcname=(
            results_csv.name
        ),
    )


    archive.write(
        summary_csv,
        arcname=(
            summary_csv.name
        ),
    )


with zipfile.ZipFile(
    temporary_bundle,
    "r",
) as archive:

    assert archive.testzip() is None


os.replace(
    temporary_bundle,
    bundle_path,
)


# ============================================================
# 15) FINAL QA
# ============================================================

assert set(
    results_df[
        "seed"
    ].tolist()
) == {
    42,
    43,
    44,
}


assert (
    results_df[
        "input_dim"
    ]
    ==
    201
).all()


assert (
    results_df[
        "checkpoint_gap"
    ]
    <=
    1e-12
).all()


print(
    "\n"
    +
    "=" * 90
)

print(
    " FINAL RDKIT-ONLY PACKAGE READY"
)

print(
    "=" * 90
)


print(
    "Results CSV:"
)

print(
    results_csv
)


print(
    "\nMean ± SD CSV:"
)

print(
    summary_csv
)


print(
    "\n3-seed ZIP:"
)

print(
    bundle_path
)


print(
    "\n Seed 42 complete"
)

print(
    " Seed 43 complete"
)

print(
    " Seed 44 complete"
)

print(
    " Input dim = 201 for all runs"
)

print(
    " Same graph recipe = 268,244 edges"
)

print(
    " Same architecture / optimizer / weighting"
)

print(
    " Calibration-only checkpoint selection"
)

print(
    " Test used only after checkpoint selection"
)

print(
    " Mean ± sample SD generated"
)

print(
    " Individual ZIPs + combined ZIP generated"
)

Source cell: 76.


In [ ]:
# ============================================================
# MORGAN + RDKIT FEATURE ABLATION
# RUN SEEDS 42 + 43 + 44 SEQUENTIALLY
# ============================================================
#
# Uses the EXACT successful controlled feature-ablation trainer
# preserved from Morgan-only Seed 42.
#
# Changes ONLY:
#
#     FEATURE_MODE = "morgan_rdkit"
#     SEED = 42 / 43 / 44
#
# EVERYTHING ELSE REMAINS IDENTICAL:
#
# - Exact original Train / Calibration / Test splits
# - 84 immutable Interaction classes
# - Train-only feature preprocessing
# - Morgan: radius=2, 2048 bits
# - RDKit: 201 Train-only selected/scaled descriptors
# - Combined input dim = 2249
# - 134,122 unique ordered Train edges
# - Reverse edges added
# - Reciprocal duplicates retained
# - Final graph = 268,244 edges
# - Same GCN architecture
# - Same ordered Drug1 -> Drug2 concat
# - Same class weights
# - Same optimizer / LR / scheduler
# - Same batch size
# - Same early stopping
# - Same Calibration Macro84 checkpoint selection
# - Test evaluated ONLY after checkpoint selection
# - Auto-resume enabled
#
# ============================================================


# ============================================================
# 1) IMPORTS
# ============================================================

import ast
import gc
import re
import json
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from IPython.display import display, FileLink


# ============================================================
# 2) PATHS
# ============================================================

FEATURE_ROOT = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_CONTROLLED_V1"
)

MORGAN_ROOT = (
    RUN_ROOT
    /
    "morgan_only"
)

COMBINED_ROOT = (
    RUN_ROOT
    /
    "morgan_rdkit"
)

COMBINED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


COMBINED_CACHE = (
    FEATURE_ROOT
    /
    "molecular_features_morgan_rdkit.npz"
)


REFERENCE_FILE = (
    MORGAN_ROOT
    /
    "reference_successful_morgan_seed42_trainer.txt"
)


# ============================================================
# 3) VERIFY COMBINED FEATURE CACHE
# ============================================================

assert COMBINED_CACHE.is_file(), (
    "Missing Morgan+RDKit feature cache:\n"
    f"{COMBINED_CACHE}\n\n"
    "Do NOT retrain yet. Cell A should already have created it."
)


with np.load(
    COMBINED_CACHE,
    allow_pickle=True,
) as data:

    combined_shape = (
        data["x"].shape
    )

    all_drugs = (
        data["all_drugs"]
        .astype(str)
    )

    train_indices = (
        data["train_drug_indices"]
        .astype(np.int64)
    )


assert combined_shape == (
    1705,
    2249,
), (
    "Unexpected Morgan+RDKit shape:",
    combined_shape,
)


assert len(
    all_drugs
) == 1705


assert len(
    set(
        all_drugs.tolist()
    )
) == 1705


assert len(
    train_indices
) == 1675


print("=" * 90)
print("MORGAN + RDKIT INPUT VERIFIED")
print("=" * 90)

print(
    "Feature shape:",
    combined_shape,
)

print(
    "Morgan dimensions:",
    2048,
)

print(
    "RDKit dimensions:",
    201,
)

print(
    "Combined dimensions:",
    2249,
)

print(
    "Total drugs:",
    len(all_drugs),
)

print(
    "Train-fit drugs:",
    len(train_indices),
)

print(
    " Combined Train-only cache ready."
)


# ============================================================
# 4) RECOVER EXACT SUCCESSFUL REFERENCE TRAINER
# ============================================================
#
# Preferred:
# reference_successful_morgan_seed42_trainer.txt
#
# Fallback:
# current notebook history.
#
# ============================================================

source42 = None


if REFERENCE_FILE.is_file():

    source42 = (
        REFERENCE_FILE.read_text(
            encoding="utf-8"
        )
    )

    print(
        "\n Using preserved successful "
        "Morgan Seed-42 reference trainer:"
    )

    print(
        REFERENCE_FILE
    )


else:

    print(
        "\nReference trainer not found."
    )

    print(
        "Trying notebook history..."
    )


    shell = get_ipython()

    assert shell is not None, (
        "Run this in the same Kaggle notebook/session."
    )


    history_cells = list(
        shell.history_manager.input_hist_raw
    )


    candidates = []


    for history_index, source in enumerate(
        history_cells
    ):

        if not isinstance(
            source,
            str,
        ):
            continue


        try:

            tree = ast.parse(
                source
            )

        except SyntaxError:

            continue


        class_names = {

            node.name

            for node in tree.body

            if isinstance(
                node,
                ast.ClassDef,
            )
        }


        if (
            "GCNDrugEncoder"
            not in class_names
        ):

            continue


        if (
            "FinalConcatOnlyGCN"
            not in class_names
        ):

            continue


        feature_mode_value = None
        seed_value = None


        for node in tree.body:

            if not isinstance(
                node,
                ast.Assign,
            ):
                continue


            if len(
                node.targets
            ) != 1:
                continue


            target = (
                node.targets[0]
            )


            if not isinstance(
                target,
                ast.Name,
            ):
                continue


            if target.id == "FEATURE_MODE":

                if (
                    isinstance(
                        node.value,
                        ast.Constant,
                    )
                    and
                    isinstance(
                        node.value.value,
                        str,
                    )
                ):

                    feature_mode_value = (
                        node.value.value
                    )


            if target.id == "SEED":

                if (
                    isinstance(
                        node.value,
                        ast.Constant,
                    )
                    and
                    isinstance(
                        node.value.value,
                        int,
                    )
                ):

                    seed_value = (
                        node.value.value
                    )


        if (
            feature_mode_value
            !=
            "morgan_only"
        ):

            continue


        if seed_value != 42:

            continue


        required_terms = [

            "FEATURE_ABLATION_CONTROLLED_V1",

            "TRAIN_ORDERED_UNIQUE_THEN_REVERSE",

            "134122",

            "268244",

            "FinalConcatOnlyGCN",

            "Calibration Macro-F1 over 84 classes",

            "best_checkpoint.pt",

            "test_predictions.csv",

            "final_result.json",
        ]


        if not all(
            term in source
            for term in required_terms
        ):

            continue


        candidates.append(
            (
                history_index,
                source,
            )
        )


    assert candidates, (
        "Could not recover successful "
        "Morgan-only Seed-42 trainer."
    )


    history_index, source42 = (
        candidates[-1]
    )


    print(
        " Recovered trainer from history index:",
        history_index,
    )


# ============================================================
# 5) STRICT REFERENCE QA
# ============================================================

tree42 = ast.parse(
    source42
)


top_level_classes = {

    node.name

    for node in tree42.body

    if isinstance(
        node,
        ast.ClassDef,
    )
}


assert (
    "GCNDrugEncoder"
    in top_level_classes
)

assert (
    "FinalConcatOnlyGCN"
    in top_level_classes
)


critical_reference_terms = [

    "TRAIN_ORDERED_UNIQUE_THEN_REVERSE",

    "134122",

    "268244",

    "ordered_concat_z1_z2",

    "weighted CrossEntropyLoss",

    "Calibration Macro-F1 over 84 classes",

    "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING",

    "best_checkpoint.pt",

    "test_predictions.csv",

    "final_result.json",
]


for term in critical_reference_terms:

    assert term in source42, (
        f"Missing protocol term: {term}"
    )


print(
    " Reference trainer passed strict protocol QA."
)


# ============================================================
# 6) HELPER — READ TOP-LEVEL ASSIGNMENTS
# ============================================================

def get_top_level_values(
    source,
):

    tree = ast.parse(
        source
    )


    feature_mode = None
    seed = None


    for node in tree.body:

        if not isinstance(
            node,
            ast.Assign,
        ):
            continue


        if len(
            node.targets
        ) != 1:
            continue


        target = (
            node.targets[0]
        )


        if not isinstance(
            target,
            ast.Name,
        ):
            continue


        if (
            target.id
            ==
            "FEATURE_MODE"
        ):

            if isinstance(
                node.value,
                ast.Constant,
            ):

                feature_mode = (
                    node.value.value
                )


        if (
            target.id
            ==
            "SEED"
        ):

            if isinstance(
                node.value,
                ast.Constant,
            ):

                seed = (
                    node.value.value
                )


    return (
        feature_mode,
        seed,
    )


reference_mode, reference_seed = (
    get_top_level_values(
        source42
    )
)


assert (
    reference_mode
    ==
    "morgan_only"
)

assert (
    reference_seed
    ==
    42
)


print(
    " Reference mode = morgan_only"
)

print(
    " Reference seed = 42"
)


# ============================================================
# 7) BUILD MORGAN+RDKIT TRAINER
# ============================================================

def build_combined_source(
    original_source,
    new_seed,
):

    assert new_seed in {
        42,
        43,
        44,
    }


    # --------------------------------------------------------
    # Replace ONLY top-level FEATURE_MODE
    # --------------------------------------------------------

    feature_pattern = (
        r'(?m)^(\s*)FEATURE_MODE\s*=\s*'
        r'["\']morgan_only["\']\s*$'
    )


    feature_matches = re.findall(
        feature_pattern,
        original_source,
    )


    assert len(
        feature_matches
    ) == 1, (
        "Expected exactly one FEATURE_MODE "
        "assignment in reference trainer."
    )


    new_source = re.sub(

        feature_pattern,

        lambda match:
            (
                match.group(1)
                +
                'FEATURE_MODE = "morgan_rdkit"'
            ),

        original_source,

        count=1,
    )


    # --------------------------------------------------------
    # Replace ONLY top-level Seed when needed
    # --------------------------------------------------------

    if new_seed != 42:

        seed_pattern = (
            r"(?m)^(\s*)SEED\s*=\s*42\s*$"
        )


        seed_matches = re.findall(
            seed_pattern,
            new_source,
        )


        assert len(
            seed_matches
        ) == 1, (
            "Expected exactly one SEED = 42 assignment."
        )


        new_source = re.sub(

            seed_pattern,

            lambda match:
                (
                    match.group(1)
                    +
                    f"SEED = {new_seed}"
                ),

            new_source,

            count=1,
        )


    # --------------------------------------------------------
    # Syntax check
    # --------------------------------------------------------

    ast.parse(
        new_source
    )


    # --------------------------------------------------------
    # Verify actual top-level values
    # --------------------------------------------------------

    mode_check, seed_check = (
        get_top_level_values(
            new_source
        )
    )


    assert (
        mode_check
        ==
        "morgan_rdkit"
    )


    assert (
        seed_check
        ==
        new_seed
    )


    # --------------------------------------------------------
    # Critical protocol must stay unchanged
    # --------------------------------------------------------

    critical_terms = [

        "TRAIN_ORDERED_UNIQUE_THEN_REVERSE",

        "134122",

        "268244",

        "FinalConcatOnlyGCN",

        "ordered_concat_z1_z2",

        "weighted CrossEntropyLoss",

        "Calibration Macro-F1 over 84 classes",

        "best_checkpoint.pt",

        "test_predictions.csv",

        "final_result.json",

        "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING",
    ]


    for term in critical_terms:

        assert term in new_source, (
            f"Protocol term disappeared: {term}"
        )


    return new_source


# ============================================================
# 8) GENERATE SCRIPTS FOR SEEDS 42/43/44
# ============================================================

generated_sources = {}
generated_scripts = {}


for seed in [
    42,
    43,
    44,
]:

    source = (
        build_combined_source(
            source42,
            seed,
        )
    )


    seed_dir = (
        COMBINED_ROOT
        /
        f"seed_{seed}"
    )


    seed_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    script_path = (
        seed_dir
        /
        f"train_morgan_rdkit_seed_{seed}.py"
    )


    if script_path.exists():

        existing_source = (
            script_path.read_text(
                encoding="utf-8"
            )
        )


        assert (
            existing_source
            ==
            source
        ), (
            f"A DIFFERENT trainer already exists "
            f"for Seed {seed}:\n"
            f"{script_path}\n\n"
            "Stopping to protect reproducibility."
        )


    else:

        script_path.write_text(
            source,
            encoding="utf-8",
        )


    generated_sources[
        seed
    ] = source


    generated_scripts[
        seed
    ] = script_path


    print(
        f" Morgan+RDKit Seed {seed} trainer prepared:"
    )

    print(
        script_path
    )


# ============================================================
# 9) RUN ONE SEED
# ============================================================

def run_one_seed(
    seed,
):

    source = (
        generated_sources[
            seed
        ]
    )


    script_path = (
        generated_scripts[
            seed
        ]
    )


    seed_dir = (
        COMBINED_ROOT
        /
        f"seed_{seed}"
    )


    result_path = (
        seed_dir
        /
        "final_result.json"
    )


    print(
        "\n"
        +
        "=" * 90
    )

    print(
        f"MORGAN + RDKIT — SEED {seed}"
    )

    print(
        "=" * 90
    )


    print(
        "Expected input dim: 2249"
    )

    print(
        "Morgan dims: 2048"
    )

    print(
        "RDKit dims: 201"
    )

    print(
        "Feature cache:"
    )

    print(
        COMBINED_CACHE
    )


    print(
        "\nFixed experimental controls:"
    )

    print(
        " Same Train/Calibration/Test rows"
    )

    print(
        " Same 84 Interaction classes"
    )

    print(
        " Same ordered Drug1→Drug2 mapping"
    )

    print(
        " Same Train-only preprocessing"
    )

    print(
        " Same 134,122 unique ordered Train edges"
    )

    print(
        " Same reverse-edge expansion"
    )

    print(
        " Same 268,244 graph edges"
    )

    print(
        " Same GCN"
    )

    print(
        " Same ordered concat"
    )

    print(
        " Same class weighting"
    )

    print(
        " Same optimizer / LR / scheduler"
    )

    print(
        " Same Calibration Macro84 model selection"
    )

    print(
        " Test only after selection"
    )

    print(
        " Auto-resume enabled"
    )


    # --------------------------------------------------------
    # Execute trainer
    # --------------------------------------------------------

    namespace = {

        "__name__":
            "__main__",

        "__file__":
            str(
                script_path
            ),
    }


    exec(
        compile(
            source,
            str(
                script_path
            ),
            "exec",
        ),
        namespace,
    )


    # --------------------------------------------------------
    # Verify result
    # --------------------------------------------------------

    assert result_path.is_file(), (
        f"Seed {seed} did not create final_result.json"
    )


    result = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    assert (
        result[
            "feature_mode"
        ]
        ==
        "morgan_rdkit"
    )


    assert (
        result[
            "seed"
        ]
        ==
        seed
    )


    assert (
        result[
            "input_dim"
        ]
        ==
        2249
    )


    assert (
        result[
            "graph_forward_unique_edges"
        ]
        ==
        134122
    )


    assert (
        result[
            "graph_total_edges"
        ]
        ==
        268244
    )


    assert (
        result[
            "train_rows"
        ]
        ==
        134249
    )


    assert (
        result[
            "calibration_rows"
        ]
        ==
        28768
    )


    assert (
        result[
            "test_rows"
        ]
        ==
        28768
    )


    assert (
        result[
            "train_fit_drugs"
        ]
        ==
        1675
    )


    assert (
        result[
            "total_drugs"
        ]
        ==
        1705
    )


    assert abs(
        result[
            "checkpoint_gap"
        ]
    ) <= 1e-12


    print(
        "\n"
        +
        "-" * 90
    )

    print(
        f" MORGAN+RDKIT SEED {seed} VERIFIED"
    )

    print(
        "-" * 90
    )


    print(
        "Completed epochs:",
        result[
            "completed_epochs"
        ],
    )


    print(
        "Best epoch:",
        result[
            "best_epoch"
        ],
    )


    print(
        "CAL Accuracy:",
        result[
            "cal_accuracy"
        ],
    )


    print(
        "CAL Macro84:",
        result[
            "cal_macro84"
        ],
    )


    print(
        "TEST Accuracy:",
        result[
            "test_accuracy"
        ],
    )


    print(
        "TEST Macro84:",
        result[
            "test_macro84"
        ],
    )


    print(
        "TEST Weighted-F1:",
        result[
            "test_weighted_f1"
        ],
    )


    del namespace

    gc.collect()

    torch.cuda.empty_cache()


    return result


# ============================================================
# 10) RUN ALL 3 SEEDS
# ============================================================

all_results = []


for seed in [
    42,
    43,
    44,
]:

    result = run_one_seed(
        seed
    )


    all_results.append(
        result
    )


# ============================================================
# 11) CREATE PER-SEED TABLE
# ============================================================

result_rows = []


for result in all_results:

    result_rows.append({

        "seed":
            result[
                "seed"
            ],

        "input_dim":
            result[
                "input_dim"
            ],

        "completed_epochs":
            result[
                "completed_epochs"
            ],

        "best_epoch":
            result[
                "best_epoch"
            ],

        "cal_accuracy":
            result[
                "cal_accuracy"
            ],

        "cal_macro84":
            result[
                "cal_macro84"
            ],

        "cal_weighted_f1":
            result[
                "cal_weighted_f1"
            ],

        "test_accuracy":
            result[
                "test_accuracy"
            ],

        "test_macro84":
            result[
                "test_macro84"
            ],

        "test_weighted_f1":
            result[
                "test_weighted_f1"
            ],

        "checkpoint_gap":
            result[
                "checkpoint_gap"
            ],
    })


results_df = (
    pd.DataFrame(
        result_rows
    )
    .sort_values(
        "seed"
    )
    .reset_index(
        drop=True
    )
)


results_csv = (
    COMBINED_ROOT
    /
    "morgan_rdkit_all_seed_results.csv"
)


results_df.to_csv(
    results_csv,
    index=False,
)


# ============================================================
# 12) MEAN ± SAMPLE SD
# ============================================================

metrics = [

    "cal_accuracy",

    "cal_macro84",

    "cal_weighted_f1",

    "test_accuracy",

    "test_macro84",

    "test_weighted_f1",
]


summary_record = {

    "feature_mode":
        "morgan_rdkit",

    "n_seeds":
        3,

    "seeds":
        "42,43,44",

    "input_dim":
        2249,
}


for metric in metrics:

    values = (
        results_df[
            metric
        ]
        .to_numpy(
            dtype=float
        )
    )


    mean_value = float(
        values.mean()
    )


    sd_value = float(
        values.std(
            ddof=1
        )
    )


    summary_record[
        f"{metric}_mean"
    ] = mean_value


    summary_record[
        f"{metric}_sd"
    ] = sd_value


    summary_record[
        f"{metric}_paper"
    ] = (
        f"{mean_value * 100:.4f} "
        f"± "
        f"{sd_value * 100:.4f}%"
    )


summary_df = pd.DataFrame(
    [
        summary_record
    ]
)


summary_csv = (
    COMBINED_ROOT
    /
    "morgan_rdkit_mean_sd_summary.csv"
)


summary_df.to_csv(
    summary_csv,
    index=False,
)


# ============================================================
# 13) SHOW FINAL 3-SEED RESULTS
# ============================================================

print(
    "\n"
    +
    "=" * 90
)

print(
    " MORGAN + RDKIT — ALL 3 SEEDS COMPLETE"
)

print(
    "=" * 90
)


display(
    results_df
)


print(
    "\n"
    +
    "=" * 90
)

print(
    "MORGAN + RDKIT — MEAN ± SAMPLE SD"
)

print(
    "=" * 90
)


display(
    summary_df[
        [
            "feature_mode",

            "cal_macro84_paper",

            "test_accuracy_paper",

            "test_macro84_paper",

            "test_weighted_f1_paper",
        ]
    ]
)


# ============================================================
# 14) CREATE COMBINED 3-SEED ZIP
# ============================================================

bundle_path = Path(
    "/kaggle/working/"
    "MORGAN_RDKIT_3SEEDS_42_43_44.zip"
)


temporary_bundle = (
    bundle_path.with_suffix(
        ".zip.tmp"
    )
)


if temporary_bundle.exists():

    temporary_bundle.unlink()


with zipfile.ZipFile(

    temporary_bundle,

    "w",

    compression=zipfile.ZIP_STORED,

) as archive:

    # --------------------------------------------------------
    # Add individual run ZIPs
    # --------------------------------------------------------

    for seed in [
        42,
        43,
        44,
    ]:

        individual_zip = (
            COMBINED_ROOT
            /
            (
                "FEATURE_ABLATION_"
                "MORGAN_RDKIT_"
                f"SEED_{seed}_COMPLETE.zip"
            )
        )


        assert individual_zip.is_file(), (
            f"Missing individual run ZIP:\n"
            f"{individual_zip}"
        )


        archive.write(
            individual_zip,
            arcname=(
                individual_zip.name
            ),
        )


    # --------------------------------------------------------
    # Add summary CSVs
    # --------------------------------------------------------

    archive.write(
        results_csv,
        arcname=(
            results_csv.name
        ),
    )


    archive.write(
        summary_csv,
        arcname=(
            summary_csv.name
        ),
    )


with zipfile.ZipFile(
    temporary_bundle,
    "r",
) as archive:

    assert archive.testzip() is None


os.replace(
    temporary_bundle,
    bundle_path,
)


# ============================================================
# 15) FINAL STRICT QA
# ============================================================

assert set(
    results_df[
        "seed"
    ].tolist()
) == {
    42,
    43,
    44,
}


assert (
    results_df[
        "input_dim"
    ]
    ==
    2249
).all()


assert (
    results_df[
        "checkpoint_gap"
    ]
    <=
    1e-12
).all()


# ============================================================
# 16) OPTIONAL THREE-WAY FEATURE ABLATION SUMMARY
# ============================================================
#
# If previous Morgan-only and RDKit-only result files are present,
# produce the final 3-way feature comparison automatically.
#
# ============================================================

MORGAN_RESULTS = (
    RUN_ROOT
    /
    "morgan_only"
    /
    "morgan_only_all_seed_results.csv"
)


# Older Morgan run may only have 43/44 summary, so rebuild from
# final_result.json files if necessary.

def load_feature_results(
    feature_mode,
):

    feature_dir = (
        RUN_ROOT
        /
        feature_mode
    )


    rows = []


    for seed in [
        42,
        43,
        44,
    ]:

        result_file = (
            feature_dir
            /
            f"seed_{seed}"
            /
            "final_result.json"
        )


        if not result_file.is_file():

            return None


        row = json.loads(
            result_file.read_text(
                encoding="utf-8"
            )
        )


        rows.append(
            row
        )


    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "seed"
        )
        .reset_index(
            drop=True
        )
    )


morgan_df = load_feature_results(
    "morgan_only"
)


rdkit_df = load_feature_results(
    "rdkit_only"
)


combined_df = load_feature_results(
    "morgan_rdkit"
)


if (
    morgan_df is not None
    and
    rdkit_df is not None
    and
    combined_df is not None
):

    comparison_rows = []


    feature_sets = [

        (
            "Morgan-only",
            "morgan_only",
            morgan_df,
        ),

        (
            "RDKit-only",
            "rdkit_only",
            rdkit_df,
        ),

        (
            "Morgan + RDKit",
            "morgan_rdkit",
            combined_df,
        ),
    ]


    for display_name, mode, df in feature_sets:

        record = {

            "feature_set":
                display_name,

            "feature_mode":
                mode,

            "input_dim":
                int(
                    df[
                        "input_dim"
                    ].iloc[0]
                ),

            "seeds":
                "42,43,44",
        }


        for metric in [

            "cal_macro84",

            "test_accuracy",

            "test_macro84",

            "test_weighted_f1",
        ]:

            values = (
                df[
                    metric
                ]
                .to_numpy(
                    dtype=float
                )
            )


            mean_value = float(
                values.mean()
            )


            sd_value = float(
                values.std(
                    ddof=1
                )
            )


            record[
                f"{metric}_mean"
            ] = mean_value


            record[
                f"{metric}_sd"
            ] = sd_value


            record[
                f"{metric}_paper"
            ] = (
                f"{mean_value * 100:.4f} "
                f"± "
                f"{sd_value * 100:.4f}%"
            )


        comparison_rows.append(
            record
        )


    feature_comparison = pd.DataFrame(
        comparison_rows
    )


    comparison_csv = (
        RUN_ROOT
        /
        "FINAL_FEATURE_ABLATION_3WAY_MEAN_SD.csv"
    )


    feature_comparison.to_csv(
        comparison_csv,
        index=False,
    )


    print(
        "\n"
        +
        "=" * 90
    )

    print(
        "FINAL THREE-WAY FEATURE ABLATION"
    )

    print(
        "=" * 90
    )


    display(
        feature_comparison[
            [
                "feature_set",

                "input_dim",

                "cal_macro84_paper",

                "test_accuracy_paper",

                "test_macro84_paper",

                "test_weighted_f1_paper",
            ]
        ]
    )


    print(
        "\nThree-way comparison saved:"
    )

    print(
        comparison_csv
    )


else:

    print(
        "\n Three-way comparison not created because "
        "one or more historical feature-set result JSONs "
        "were not found."
    )


# ============================================================
# 17) FINAL REPORT
# ============================================================

print(
    "\n"
    +
    "=" * 90
)

print(
    " FINAL MORGAN + RDKIT PACKAGE READY"
)

print(
    "=" * 90
)


print(
    "Per-seed results:"
)

print(
    results_csv
)


print(
    "\nMean ± SD:"
)

print(
    summary_csv
)


print(
    "\n3-seed ZIP:"
)

print(
    bundle_path
)


print(
    "\nQA:"
)

print(
    " Seed 42 complete"
)

print(
    " Seed 43 complete"
)

print(
    " Seed 44 complete"
)

print(
    " Input dim = 2249"
)

print(
    " Morgan = 2048 features"
)

print(
    " RDKit = 201 Train-only descriptors"
)

print(
    " Same original Interaction splits"
)

print(
    " Same 84 Interaction labels"
)

print(
    " Same ordered Drug1→Drug2 representation"
)

print(
    " Same 134,122 unique Train edges"
)

print(
    " Same 268,244 final graph edges"
)

print(
    " Same model / class weights / optimizer"
)

print(
    " Calibration-only checkpoint selection"
)

print(
    " Test evaluated only after model selection"
)

print(
    " Auto-resume enabled"
)

print(
    " Mean ± sample SD generated"
)

print(
    " ZIP integrity passed"
)


display(
    FileLink(
        str(
            bundle_path.relative_to(
                "/kaggle/working"
            )
        )
    )
)